<a href="https://colab.research.google.com/github/AlperYildirim1/Time-Series-PRISM/blob/main/ETTm1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q x-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.4 MB/s eta 0:00:00


## Data

In [ ]:
import os
import math
import json
import random
import logging
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from x_transformers import Encoder
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

# 0. SETUP SEED
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device}")


SAVE_DIR = '/content/drive/MyDrive/PRISM_ETTm2'
LOG_DIR  = '/content/drive/MyDrive/PRISM_Logs_ETTm2'
TB_DIR   = '/content/drive/MyDrive/PRISM_TensorBoard_ETTm2'


os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def get_logger(run_name):
    logger = logging.getLogger(run_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fh = logging.FileHandler(os.path.join(LOG_DIR, f"{run_name}_{timestamp}.log"))
    fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(ch)

    return logger

# 1. DATA — ETTm2
print("\n⏳ Loading ETTm2 dataset...")
url = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/ETT-small/ETTm2.csv"
df = pd.read_csv(url)

date_col = df.columns[0]
cols_data = df.drop(columns=[date_col]).values.astype(np.float32)
n_total = len(cols_data)
n_channels = cols_data.shape[1]

print(f"✅ Loaded {n_total} rows, {n_channels} channels: {df.columns[1:].tolist()}")

# ---------------------------------------------------------
# CRITICAL: ETT uses HARDCODED month-based splits, NOT %
# 12 months train / 4 months val / 4 months test
# These are the EXACT boundaries from TSLib data_loader.py
# ---------------------------------------------------------
train_end = 12 * 30 * 24 * 4      # 34560
val_end   = train_end + 4 * 30 * 24 * 4   # 46080
test_end  = val_end + 4 * 30 * 24 * 4     # 57600

print(f"   Splits: train=0:{train_end} | val=:{val_end} | test=:{test_end}")
print(f"   Unused tail rows: {n_total - test_end}")

scaler = StandardScaler()
scaler.fit(cols_data[:train_end])
all_data = scaler.transform(cols_data)

seq_len = 336
batch_size = 128  # Small dataset + few channels → bigger batch is fine

# We'll loop over all 4 horizons
pred_lens = [96, 192, 336, 720]


🖥️ Device: cuda

⏳ Loading ETTm2 dataset...
✅ Loaded 69680 rows, 7 channels: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
   Splits: train=0:34560 | val=:46080 | test=:57600
   Unused tail rows: 12080


## Classes

In [ ]:

class TSDataset(Dataset):
    def __init__(self, data, seq_len=336, pred_len=96):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len
    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1
    def __getitem__(self, idx):
        s = idx + self.seq_len
        return torch.FloatTensor(self.data[idx:s]), torch.FloatTensor(self.data[s:s + self.pred_len])

# 2. MODEL COMPONENTS
class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))
    def forward(self, x, mode):
        if mode == 'norm':
            self.mean = x.mean(dim=1, keepdim=True).detach()
            self.stdev = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self.mean) / self.stdev
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x * self.stdev + self.mean

class ContinuousPhaseInjector(nn.Module):
    def __init__(self, patch_len, d_model, period, max_patches=1024):
        super().__init__()
        self.d_model = d_model
        self.particle_proj = nn.Linear(patch_len, d_model)
        self.dynamic_angle_proj = nn.Linear(patch_len, d_model)
        base_freq = 2 * math.pi / period
        k = torch.arange(1, d_model + 1, dtype=torch.float32)
        static_freqs = k * base_freq
        t = torch.arange(max_patches, dtype=torch.float32)
        self.register_buffer("static_angles", torch.outer(t, static_freqs))

    def forward(self, x):
        particle_src = self.particle_proj(x)
        with torch.amp.autocast('cuda', enabled=False):
            x_32 = x.float()
            w32 = self.dynamic_angle_proj.weight.float()
            b32 = self.dynamic_angle_proj.bias.float() if self.dynamic_angle_proj.bias is not None else None
            dynamic_angles = F.linear(x_32, w32, b32)
            L = x.shape[1]
            static_angles_slice = self.static_angles[:L].unsqueeze(0)
            total_angles = dynamic_angles + static_angles_slice
            wave_src = torch.polar(torch.ones_like(total_angles), total_angles)
        return wave_src, particle_src

class HyenaNeuralFilter(nn.Module):
    def __init__(self, d_model, max_len=1024, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        freqs = torch.exp(torch.arange(0, hidden_dim, 2, dtype=torch.float32) * -(math.log(10000.0) / hidden_dim))
        t = torch.linspace(0, 1, steps=max_len).unsqueeze(-1)
        emb = torch.cat([torch.sin(t * freqs), torch.cos(t * freqs)], dim=-1)
        self.register_buffer("emb", emb)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2),
        )

    def forward(self, L):
        emb_L = self.emb[:L]
        out_32 = self.mlp(emb_L).view(L, self.d_model, 2).float()
        return torch.complex(out_32[..., 0], out_32[..., 1])

class RobustPhaseNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))
    def forward(self, z):
        z_32 = z.to(torch.complex64)
        mag = torch.abs(z_32)
        new_mag = F.relu(mag + self.b.float())
        phase = z_32 / (mag + 1e-6)
        return (new_mag * phase).to(z.dtype)

class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.filter_len = max_len
        self.neural_filter = HyenaNeuralFilter(d_model, max_len=max_len)
        self.gate_proj = nn.Linear(d_model * 2, d_model * 2)
        self.mix_real = nn.Linear(d_model, d_model)
        self.mix_imag = nn.Linear(d_model, d_model)
        self.out_real = nn.Linear(d_model, d_model)
        self.out_imag = nn.Linear(d_model, d_model)
        self.activation = ModReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)
        with torch.amp.autocast('cuda', enabled=False):
            x_32 = x_norm.to(torch.complex64)
            B, L, D = x_32.shape
            eff_L = min(L, self.filter_len)
            x_freq = torch.fft.fft(x_32, n=eff_L, dim=1, norm='ortho')
            h = self.neural_filter(eff_L).unsqueeze(0).to(torch.complex64)
            x_filtered = x_freq * h
            x_time = torch.fft.ifft(x_filtered, n=eff_L, dim=1, norm='ortho')
            x_cat = torch.cat([x_32.real, x_32.imag], dim=-1)
            gate_w = self.gate_proj.weight.to(torch.float32)
            gate_b = self.gate_proj.bias.to(torch.float32)
            gate_out = F.linear(x_cat, gate_w, gate_b)
            gates = torch.sigmoid(gate_out)
            g_r, g_i = gates.chunk(2, dim=-1)
            x_gated_32 = torch.complex(x_time.real * g_r, x_time.imag * g_i)
            x_gated = x_gated_32.to(x.dtype)

        mr, mi = self.mix_real, self.mix_imag
        r_part = (mr(x_gated.real) - mi(x_gated.imag)).float()
        i_part = (mr(x_gated.imag) + mi(x_gated.real)).float()
        x_mixed = torch.complex(r_part, i_part)
        x_act = self.activation(x_mixed)

        or_, oi = self.out_real, self.out_imag
        r_out = (or_(x_act.real) - oi(x_act.imag)).float()
        i_out = (or_(x_act.imag) + oi(x_act.real)).float()
        out = torch.complex(r_out, i_out)
        return out + residual


class PRISMSkipTransformer(nn.Module):
    def __init__(self, seq_len=336, pred_len=96, patch_len=16, stride=8, d_model=64,
                 prism_depth=1, transformer_depth=1, period=3.0, dropout=0.2, n_channels=7):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model
        self.num_patches = (seq_len - patch_len) // stride + 2

        self.revin = RevIN(n_channels)
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.phase_injector = ContinuousPhaseInjector(patch_len, d_model, period)

        self.prism = nn.Sequential(*[GatedHarmonicConvolution(d_model, self.num_patches, dropout) for _ in range(prism_depth)])
        self.prism_norm = RobustPhaseNorm(d_model)

        self.bridge_proj = nn.Linear(d_model * 2, d_model)
        self.bridge_norm = nn.LayerNorm(d_model)
        self.skip_norm = nn.LayerNorm(d_model)

        transformer_heads = max(1, d_model // 8)
        self.transformer = Encoder(
            dim=d_model, depth=transformer_depth, heads=transformer_heads,
            attn_flash=True, rotary_pos_emb=True, use_rmsnorm=True, ff_glu=True,
            attn_dropout=dropout, ff_dropout=dropout
        )

        self.flatten_dim = self.num_patches * d_model
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.flatten_dim, pred_len)

    def forward(self, x):
        x = self.revin(x, 'norm')
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)

        x = x.transpose(1, 2)
        x = F.pad(x, (0, self.stride), 'replicate')
        patches = x.unfold(-1, self.patch_len, self.stride).squeeze(1)

        raw_embed = self.patch_embed(patches)
        wave_src, _ = self.phase_injector(patches)

        prism_out = self.prism_norm(self.prism(wave_src))

        cat = torch.cat([prism_out.real, prism_out.imag], dim=-1)
        bridged = self.bridge_norm(self.bridge_proj(cat))

        transformer_input = self.skip_norm(bridged + raw_embed)
        refined = self.transformer(transformer_input)

        flattened = self.head_drop(refined.reshape(refined.shape[0], -1))
        out = self.head(flattened).unsqueeze(-1)

        out = out.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return self.revin(out, 'denorm')


## Training

In [ ]:

# 3. CONFIGS — ETTm2
period_patches = 12.0 # Daily period for 15 min data

configs = [
    {
        "name": "ETTm2_d16_depth1_1",
        "d_model": 16, "prism_depth": 1, "transformer_depth": 1,
        "dropout": 0.3, "lr": 2e-4, "epochs": 50,
    },

]

# 4. TRAINING — All 4 horizons × all configs
all_results = {}

for pred_len in pred_lens:
    print(f"\n{'#'*60}")
    print(f"  HORIZON: {seq_len} → {pred_len}")
    print(f"{'#'*60}")

    # Rebuild dataloaders for this horizon
    train_slice = all_data[0:train_end]
    val_slice   = all_data[train_end - seq_len : val_end]
    test_slice  = all_data[val_end - seq_len : test_end]

    g = torch.Generator().manual_seed(SEED)
    loader_kwargs = dict(num_workers=4, pin_memory=True, persistent_workers=True)

    train_loader = DataLoader(
        TSDataset(train_slice, seq_len, pred_len),
        batch_size=batch_size, shuffle=True, drop_last=True,
        generator=g, worker_init_fn=lambda w: np.random.seed(SEED + w),
        **loader_kwargs
    )
    val_loader = DataLoader(
        TSDataset(val_slice, seq_len, pred_len),
        batch_size=batch_size, shuffle=False, **loader_kwargs
    )
    test_loader = DataLoader(
        TSDataset(test_slice, seq_len, pred_len),
        batch_size=batch_size, shuffle=False, **loader_kwargs
    )

    for cfg in configs:
        name = f"{cfg['name']}_h{pred_len}_seed{SEED}"
        logger = get_logger(name)
        writer = SummaryWriter(log_dir=os.path.join(TB_DIR, f"{name}_{timestamp}"))

        logger.info(f"{'='*60}")
        logger.info(f"RUN: {name}")
        logger.info(f"Config: {json.dumps(cfg, indent=2)}")
        logger.info(f"Dataset: ETTm2 | Channels: {n_channels} | Seq: {seq_len} → Pred: {pred_len}")
        logger.info(f"Split: train=0:{train_end} | val=:{val_end} | test=:{test_end} (TSLib hardcoded)")
        logger.info(f"{'='*60}")

        model = PRISMSkipTransformer(
            seq_len=seq_len, pred_len=pred_len,
            d_model=cfg["d_model"], prism_depth=cfg["prism_depth"],
            transformer_depth=cfg["transformer_depth"], period=period_patches,
            dropout=cfg["dropout"], n_channels=n_channels,
        ).to(device)

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        logger.info(f"Parameters: {n_params:,}")
        writer.add_text("config", json.dumps(cfg, indent=2), 0)

        optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0)
        criterion = nn.MSELoss()
        best_val, no_improve, patience = float('inf'), 0, 10
        best_ckpt_path = os.path.join(SAVE_DIR, f"{name}_best.pt")
        global_step = 0

        for epoch in range(cfg["epochs"]):
            model.train()
            tl = 0
            pbar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}", leave=False)
            for x, y in pbar:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    loss = criterion(model(x), y)
                loss.backward()

                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                writer.add_scalar("train/grad_norm", grad_norm.item(), global_step)
                writer.add_scalar("train/batch_loss", loss.item(), global_step)

                optimizer.step()
                tl += loss.item()
                global_step += 1
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            avg_t = tl / len(train_loader)

            model.eval()
            vl = 0
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    vl += criterion(model(x), y).item()

            avg_v = vl / len(val_loader)

            writer.add_scalars("loss/epoch", {"train": avg_t, "val": avg_v}, epoch + 1)
            writer.add_scalar("loss/train", avg_t, epoch + 1)
            writer.add_scalar("loss/val", avg_v, epoch + 1)
            writer.add_scalar("meta/learning_rate", optimizer.param_groups[0]['lr'], epoch + 1)

            if avg_v < best_val:
                best_val = avg_v
                no_improve = 0
                tag = "🏆 (Saved)"
                torch.save(model.state_dict(), best_ckpt_path)
            else:
                no_improve += 1
                tag = ""

            logger.info(f"Ep {epoch+1:02d} | Train: {avg_t:.4f} | Val: {avg_v:.4f} | Best: {best_val:.4f} {tag}")

            if no_improve >= patience:
                logger.info(f"⏹ Early stop at epoch {epoch+1}")
                break

        # 5. EVALUATION (BATCH-WEIGHTED)
        logger.info(f"Loading best checkpoint for testing...")
        model.load_state_dict(torch.load(best_ckpt_path))
        model.eval()

        mse_sum, mae_sum, total_samples = 0.0, 0.0, 0
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                p = model(x)
                b_size = x.size(0)
                mse_sum += F.mse_loss(p, y).item() * b_size
                mae_sum += F.l1_loss(p, y).item() * b_size
                total_samples += b_size

        final_mse = mse_sum / total_samples
        final_mae = mae_sum / total_samples

        logger.info(f"✅ FINAL | Test MSE: {final_mse:.4f} | Test MAE: {final_mae:.4f}")

        writer.add_hparams(
            hparam_dict={
                "seed": SEED,
                "batch_size": batch_size,
                "seq_len": seq_len,
                "pred_len": pred_len,
                "patch_len": 16,
                "stride": 8,
                "period": period_patches,
                "d_model": cfg["d_model"],
                "prism_depth": cfg["prism_depth"],
                "transformer_depth": cfg["transformer_depth"],
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
                "weight_decay": 0,
                "patience": patience,
                "scheduler": "none",
                "n_channels": n_channels,
                "n_params": n_params,
            },
            metric_dict={
                "hparam/test_mse": final_mse,
                "hparam/test_mae": final_mae,
                "hparam/best_val": best_val,
            },
        )
        writer.close()

        result_key = f"{cfg['name']}_h{pred_len}"
        all_results[result_key] = {"mse": final_mse, "mae": final_mae, "best_val": best_val, "params": n_params}

        results_path = os.path.join(LOG_DIR, f"{name}_{timestamp}_results.json")
        with open(results_path, 'w') as f:
            json.dump(all_results[result_key] | cfg | {"pred_len": pred_len, "seed": SEED}, f, indent=2)
        logger.info(f"Results saved to {results_path}")

        del model, optimizer
        torch.cuda.empty_cache()


# 6. SUMMARY — ALL HORIZONS
print(f"\n{'='*70}")
print(f"  ETTm2 RESULTS — All Horizons")
print(f"{'='*70}")

# Published baselines
baselines = {
    "PatchTST_h96":  {"mse": 0.1111111, "mae": 0.111111111}, # placeholder

}

for h in pred_lens:
    print(f"\n  --- Horizon {h} ---")
    horizon_results = {k: v for k, v in all_results.items() if f"_h{h}" in k}
    horizon_baselines = {k: v for k, v in baselines.items() if f"_h{h}" in k}
    combined = {**horizon_results, **horizon_baselines}

    for name, v in sorted(combined.items(), key=lambda x: x[1]["mse"]):
        mse = v["mse"]
        mae = v["mae"]
        params = v.get("params", "—")
        print(f"    {name:<35} MSE: {mse:.4f}  MAE: {mae:.4f}  params: {params}")

print(f"\n{'='*70}")
print(f"📂 Checkpoints: {SAVE_DIR}")
print(f"📂 Logs:        {LOG_DIR}")
print(f"📂 TensorBoard: {TB_DIR}")

RUN: ETTm2_d16_depth1_1_h96_seed1
Config: {
  "name": "ETTm2_d16_depth1_1",
  "d_model": 16,
  "prism_depth": 1,
  "transformer_depth": 1,
  "dropout": 0.3,
  "lr": 0.0002,
  "epochs": 50
}
Dataset: ETTm2 | Channels: 7 | Seq: 336 → Pred: 96
Split: train=0:34560 | val=:46080 | test=:57600 (TSLib hardcoded)



############################################################
  HORIZON: 336 → 96
############################################################


Parameters: 90,078


Ep 01:   0%|          | 0/266 [00:00<?, ?it/s]

Ep 01 | Train: 0.2989 | Val: 0.1232 | Best: 0.1232 🏆 (Saved)


...

Ep 09:   0%|          | 0/261 [00:00<?, ?it/s]

Ep 09 | Train: 0.4638 | Val: 0.2775 | Best: 0.2706 


Ep 10:   0%|          | 0/261 [00:00<?, ?it/s]

Ep 10 | Train: 0.4594 | Val: 0.2780 | Best: 0.2706 


Ep 11:   0%|          | 0/261 [00:00<?, ?it/s]

## Else

In [ ]:

# # ENV LOGGING
# import subprocess, platform, sys

# env_info = {
#     "python": sys.version,
#     "platform": platform.platform(),
#     "cpu": platform.processor(),
#     "pytorch": torch.__version__,
# }

# if torch.cuda.is_available():
#     env_info["cuda_version"] = torch.version.cuda
#     env_info["cudnn_version"] = str(torch.backends.cudnn.version())
#     env_info["gpu_name"] = torch.cuda.get_device_name(0)
#     env_info["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
#     env_info["gpu_count"] = torch.cuda.device_count()
# else:
#     env_info["gpu"] = "None (CPU only)"

# try:
#     freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
#     env_info["pip_freeze"] = freeze.strip().split("\n")
# except Exception:
#     env_info["pip_freeze"] = "unavailable"

# env_path = os.path.join(LOG_DIR, f"environment_{timestamp}.json")
# with open(env_path, 'w') as f:
#     json.dump(env_info, f, indent=2)

# print(json.dumps({k: v for k, v in env_info.items() if k != "pip_freeze"}, indent=2))
# print(f"\nFull env saved to: {env_path}")

In [ ]:
# # Results Aggregator — run after all 3 seeds
# import os, json, glob
# import numpy as np

# LOG_DIR = '/content/drive/MyDrive/PRISM_Logs_ETTm2'

# # Collect all results JSONs
# files = glob.glob(os.path.join(LOG_DIR, "*_results.json"))
# print(f"Found {len(files)} result files\n")

# # Parse into structured dict: {(config, horizon): [results]}
# from collections import defaultdict
# grouped = defaultdict(list)

# for f in sorted(files):
#     with open(f) as fh:
#         r = json.load(fh)
#     # Extract config name without seed suffix and horizon
#     name = r.get("name", "unknown")
#     pred_len = r.get("pred_len", 0)
#     seed = r.get("seed", "?")
#     key = (name, pred_len)
#     grouped[key].append({
#         "seed": seed,
#         "mse": r["mse"],
#         "mae": r["mae"],
#         "file": os.path.basename(f),
#     })

# # Print per-seed detail + aggregated stats
# print(f"{'Config':<35} {'H':>4} {'Seed':>6} {'MSE':>8} {'MAE':>8}")
# print("-" * 70)

# for (name, h), runs in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
#     for r in sorted(runs, key=lambda x: x["seed"]):
#         print(f"{name:<35} {h:>4} {r['seed']:>6} {r['mse']:>8.4f} {r['mae']:>8.4f}")
#     print()

# # Summary table
# print(f"\n{'='*85}")
# print(f"  ETTm2 SUMMARY — mean ± std (min / max) across seeds")
# print(f"{'='*85}")
# print(f"{'Config':<30} {'H':>4} {'N':>3}  {'MSE mean±std':<20} {'MAE mean±std':<20} {'MSE range':<18}")
# print("-" * 85)

# for (name, h), runs in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
#     mses = np.array([r["mse"] for r in runs])
#     maes = np.array([r["mae"] for r in runs])
#     n = len(runs)
#     print(f"{name:<30} {h:>4} {n:>3}  "
#           f"{mses.mean():.4f}±{mses.std():.4f}      "
#           f"{maes.mean():.4f}±{maes.std():.4f}      "
#           f"{mses.min():.4f}/{mses.max():.4f}")

# # Baselines for comparison
# print(f"\n{'='*85}")
# print(f"  Published Baselines (for reference)")
# print(f"{'='*85}")
# baselines = {
#     ("PatchTST", 96): (0.370, 0.400), ("PatchTST", 192): (0.413, 0.429),
#     ("PatchTST", 336): (0.422, 0.440), ("PatchTST", 720): (0.447, 0.468),
#     ("DLinear", 96): (0.375, 0.399), ("DLinear", 192): (0.405, 0.416),
#     ("DLinear", 336): (0.439, 0.443), ("DLinear", 720): (0.472, 0.490),
# }
# print(f"{'Model':<30} {'H':>4}  {'MSE':>8} {'MAE':>8}")
# print("-" * 55)
# for (name, h), (mse, mae) in sorted(baselines.items(), key=lambda x: (x[0][0], x[0][1])):
#     print(f"{name:<30} {h:>4}  {mse:>8.4f} {mae:>8.4f}")

In [ ]:
# ==========================================
# ETTm2 Data Inspection Cell
# ==========================================
import pandas as pd
import numpy as np

url = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/ETT-small/ETTm2.csv"
df = pd.read_csv(url)

print(f"=== SHAPE: {df.shape} ===")
print(f"Date range: {df.iloc[0,0]} → {df.iloc[-1,0]}")

cols_data = df.drop(columns=['date']).values.astype(np.float32)

print(f"\n=== PERIODICITY (ACF) ===")
max_lag = 500
for i in range(cols_data.shape[1]):
    col = cols_data[:, i]
    col = (col - col.mean()) / (col.std() + 1e-8)
    n_pts = len(col)
    acf = np.correlate(col, col, mode='full')[n_pts-1:n_pts-1+max_lag]
    acf /= acf[0]
    peaks = []
    for j in range(2, max_lag - 1):
        if acf[j] > acf[j-1] and acf[j] > acf[j+1] and acf[j] > 0.1:
            peaks.append((j, acf[j]))
    peaks.sort(key=lambda x: -x[1])
    top = peaks[0] if peaks else (0, 0)
    second = peaks[1] if len(peaks) > 1 else (0, 0)
    print(f"  {df.columns[i+1]:>6}: period1={top[0]:>3} steps (r={top[1]:.3f})  period2={second[0]:>3} steps (r={second[1]:.3f})")